# Neural Video Codec — Modular Pipeline Demo

This notebook walks through each stage of the pipeline independently so you can see exactly what each module does and how they compose.

```
Input Video
    │
    ▼
┌─────────────┐
│  compress   │  ← DCVC neural codec, ROI/BG split streams
└──────┬──────┘
       │  ZIP archive (roi.bin + bg.bin + detections.json)
    ▼
┌─────────────┐
│ decompress  │  ← DCVC decode + alpha-compositing
└──────┬──────┘
       │  BGR frames
    ▼
┌─────────────┐
│   restore   │  ← RestoreUNet diffusion denoiser
└──────┬──────┘
       │  denoised BGR frames
    ▼
┌──────────────────────────────────────┐
│  upscale  (pick one)                 │
│   • bicubic   — fast Lanczos resize  │
│   • cogvideo  — CogVideoX V2V        │
│   • wan       — Wan2.1 I2V           │
└──────┬───────────────────────────────┘
       │  HR BGR frames
    ▼
 Output MP4
```

Every stage can be run **independently** — you can skip stages, start mid-pipeline, or swap upscalers.

## 0 — Setup

In [ ]:
import sys, json, zipfile, io
from pathlib import Path
import cv2
import numpy as np
import yaml
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Video, display

ROOT = Path().resolve()
sys.path.insert(0, str(ROOT))

# ── helpers ───────────────────────────────────────────────────────────────────
def show_frames(frames, titles=None, n=5, figsize=(18, 4), cmap=None):
    """Display n evenly-spaced frames side by side."""
    idxs = np.linspace(0, len(frames) - 1, min(n, len(frames)), dtype=int)
    fig, axes = plt.subplots(1, len(idxs), figsize=figsize)
    if len(idxs) == 1:
        axes = [axes]
    for ax, i in zip(axes, idxs):
        rgb = cv2.cvtColor(frames[i], cv2.COLOR_BGR2RGB)
        ax.imshow(rgb, cmap=cmap)
        ax.set_title(titles[i] if titles else f"frame {i}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def compare_frames(frame_groups, group_labels, frame_idx=None, figsize=(18, 6)):
    """
    Show the same frames from multiple groups side by side.
    frame_groups: list of frame-lists
    group_labels: label per group
    frame_idx: list of indices to show; defaults to 5 evenly-spaced
    """
    n_frames = len(frame_groups[0])
    if frame_idx is None:
        frame_idx = np.linspace(0, n_frames - 1, 5, dtype=int).tolist()
    n_groups = len(frame_groups)
    n_cols   = len(frame_idx)
    fig, axes = plt.subplots(n_groups, n_cols, figsize=figsize)
    if n_groups == 1:
        axes = [axes]
    for r, (frames, label) in enumerate(zip(frame_groups, group_labels)):
        for c, fi in enumerate(frame_idx):
            ax = axes[r][c]
            rgb = cv2.cvtColor(frames[fi], cv2.COLOR_BGR2RGB)
            ax.imshow(rgb)
            if r == 0:
                ax.set_title(f"frame {fi}", fontsize=9)
            if c == 0:
                ax.set_ylabel(label, fontsize=9, rotation=0, labelpad=60, va="center")
            ax.axis("off")
    plt.tight_layout()
    plt.show()

def read_video(path) -> list:
    """Load all BGR frames from a video file."""
    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        frames.append(f)
    cap.release()
    return frames

print("Setup OK — ROOT:", ROOT)

## 1 — Config

In [ ]:
CONFIG_PATH = ROOT / "configs" / "gpu" / "compression.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Load restoration sub-config
rest_config_path = ROOT / cfg.get("restoration", {}).get("config", "configs/gpu/restoration.yaml")
if rest_config_path.exists():
    with open(rest_config_path) as f:
        cfg["_restoration"] = yaml.safe_load(f)

print("Top-level config keys:", list(cfg.keys()))
print("\nDetection config:")
print(yaml.dump({"detection": cfg["detection"]}, default_flow_style=False))

In [ ]:
# ── Choose your input video ────────────────────────────────────────────────────
INPUT_VIDEO = ROOT / "data" / "bird1.mp4"   # change to your video

cap = cv2.VideoCapture(str(INPUT_VIDEO))
fps         = cap.get(cv2.CAP_PROP_FPS) or 30.0
total       = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Video: {INPUT_VIDEO.name}")
print(f"  Resolution : {width}×{height}")
print(f"  Frames     : {total}")
print(f"  FPS        : {fps:.1f}")
print(f"  Duration   : {total/fps:.1f}s")

## 2 — Stage 1: Compress

The compressor:
1. Runs animal detection (MegaDetector or YOLO11) on the video
2. Splits each frame into ROI (detected regions) and Background streams
3. Encodes ROI at high quality (QP 63) and BG aggressively (QP 5)
4. Packs everything into a ZIP archive

In [ ]:
from src.compression import compress_video

archive_bytes = compress_video(str(INPUT_VIDEO), cfg)

input_size_mb   = INPUT_VIDEO.stat().st_size / 1e6
archive_size_mb = len(archive_bytes) / 1e6
ratio           = input_size_mb / archive_size_mb

print(f"Input  : {input_size_mb:.2f} MB")
print(f"Archive: {archive_size_mb:.2f} MB")
print(f"Ratio  : {ratio:.1f}×")

# Show what's inside the archive
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as arc:
    for info in arc.infolist():
        print(f"  {info.filename:<30} {info.file_size/1e3:8.1f} KB")

In [ ]:
# Peek at the detections that were saved into the archive
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as arc:
    detections_raw = json.loads(arc.read("detections.json"))
    meta           = json.loads(arc.read("meta.json"))

n_detected = sum(1 for v in detections_raw.values() if v)
n_total    = len(detections_raw)

print(f"Detected frames : {n_detected} / {n_total}")
print(f"Meta: {json.dumps(meta, indent=2)}")

# Visualise detections on a few raw frames
raw_frames = read_video(INPUT_VIDEO)
vis_frames = []
for fi, frame in enumerate(raw_frames):
    fc = frame.copy()
    for b in (detections_raw.get(fi) or detections_raw.get(str(fi)) or []):
        cv2.rectangle(fc, (b["x1"], b["y1"]), (b["x2"], b["y2"]), (0, 255, 0), 2)
    vis_frames.append(fc)

show_frames(vis_frames, n=5, figsize=(18, 4))
print("Green boxes = detected animal ROI")

## 3 — Stage 2: Decompress

The decompressor:
1. Unpacks the ZIP archive
2. Decodes the ROI and BG bitstreams with DCVC
3. Alpha-composites ROI over BG using the saved detection masks

In [ ]:
from src.decompression import decompress_archive

decomp = decompress_archive(archive_bytes, cfg)
decomp_frames = decomp["frames"]
detections    = decomp["detections"]   # {frame_idx: [bbox, ...]}
decomp_fps    = decomp["fps"]

print(f"Decompressed: {len(decomp_frames)} frames @ {decomp_fps:.1f} fps")
print(f"Frame shape : {decomp_frames[0].shape}")

compare_frames(
    [raw_frames, decomp_frames],
    ["Original", "Decompressed"],
    figsize=(18, 6)
)

## 4 — Stage 3: Restore

The RestoreUNet diffusion model:
- Denoises compression artifacts using DDIM sampling
- Uses a **temporal window** of T frames so neighbouring frames inform the denoising
- In ROI regions the blend strength is configurable — preserve animal detail

In [ ]:
from src.restoration import restore_frames

restore_cfg = cfg.get("_restoration", cfg)   # uses restoration.yaml sub-config

restored_frames = restore_frames(
    decomp_frames,
    restore_cfg,
    detections,
    width  = decomp_frames[0].shape[1],
    height = decomp_frames[0].shape[0],
)

print(f"Restored: {len(restored_frames)} frames")

compare_frames(
    [raw_frames, decomp_frames, restored_frames],
    ["Original", "Decompressed", "Restored"],
    figsize=(18, 8)
)

## 5 — Stage 4a: Upscale — Bicubic (fast baseline)

In [ ]:
out_w = cfg.get("cogvideo_upscaling", {}).get("out_w", 720)
out_h = cfg.get("cogvideo_upscaling", {}).get("out_h", 480)

bicubic_frames = [
    cv2.resize(f, (out_w, out_h), interpolation=cv2.INTER_LANCZOS4)
    for f in restored_frames
]

print(f"Bicubic upscale: {restored_frames[0].shape[:2][::-1]} → {(out_w, out_h)}")

compare_frames(
    [raw_frames, decomp_frames, bicubic_frames],
    ["Original", "Decompressed", "Bicubic ×2"],
    figsize=(18, 8)
)

## 5 — Stage 4b: Upscale — CogVideoX V2V (temporal diffusion)

CogVideoX Video-to-Video:
- Encodes the full video, adds controlled noise (`strength`), denoises with 3D spatiotemporal attention
- Processes in overlapping chunks; blends at boundaries
- When `detections` are passed, the original bicubic frames are **pasted on top** in ROI regions so the model never hallucinates the animal

In [ ]:
from src.upscaling.cogvideo_upscaler import CogVideoUpscaler

cog_cfg = cfg.get("cogvideo_upscaling", {})
print("CogVideoX config:")
print(yaml.dump(cog_cfg, default_flow_style=False))

In [ ]:
cogvideo_upscaler = CogVideoUpscaler(cog_cfg)

# Pass detections so that ROI areas keep original bicubic frames
# (prevents model from hallucinating the subject)
cogvideo_frames = cogvideo_upscaler.upscale_sequence(
    restored_frames,
    detections=detections,
)

print(f"CogVideoX output: {len(cogvideo_frames)} frames, shape {cogvideo_frames[0].shape}")

In [ ]:
compare_frames(
    [raw_frames, decomp_frames, bicubic_frames, cogvideo_frames],
    ["Original", "Decompressed", "Bicubic", "CogVideoX"],
    figsize=(18, 12)
)

## 5 — Stage 4c: Upscale — Wan2.1 I2V (generative)

Wan2.1 Image-to-Video:
- Conditions each chunk on its first frame, generates HR video
- Good for short clips; less temporally stable across chunk boundaries
- Use `generate_wan_bookends.py` to generate the full video from a single endpoint frame

In [ ]:
from src.upscaling.wan_upscaler import WanUpscaler

wan_cfg = cfg.get("wan_upscaling", {})
print("Wan2.1 config:")
print(yaml.dump(wan_cfg, default_flow_style=False))

In [ ]:
wan_upscaler = WanUpscaler(wan_cfg)
wan_frames   = wan_upscaler.upscale_sequence(restored_frames)

print(f"Wan output: {len(wan_frames)} frames, shape {wan_frames[0].shape}")

compare_frames(
    [raw_frames, bicubic_frames, cogvideo_frames, wan_frames],
    ["Original", "Bicubic", "CogVideoX", "Wan2.1"],
    figsize=(18, 12)
)

## 6 — Save Outputs

In [ ]:
from src.postprocessing.video_assembler import assemble_video

out_dir = ROOT / "outputs" / "notebook_demo"
out_dir.mkdir(parents=True, exist_ok=True)

stem = INPUT_VIDEO.stem

saves = {
    "decomp":    decomp_frames,
    "restored":  restored_frames,
    "bicubic":   bicubic_frames,
    "cogvideo":  cogvideo_frames,
    # "wan":     wan_frames,   # uncomment if you ran the Wan cell
}

for tag, frames in saves.items():
    out_path = out_dir / f"{stem}_{tag}.mp4"
    assemble_video(iter(frames), out_path, fps=decomp_fps)
    print(f"Saved {len(frames)} frames → {out_path}")

## 7 — Just Upscale (skip compress/decompress/restore)

If you already have a decompressed video (e.g. from `run_pipeline.py --stages decompress`),
you can run **only the upscale stage** directly.

In [ ]:
# ── Load existing decompressed video + detections from archive ────────────────
EXISTING_VIDEO    = ROOT / "outputs" / "compression" / "bird1_decompressed.mp4"
EXISTING_ARCHIVE  = ROOT / "outputs" / "compression" / "bird1.zip"

existing_frames = read_video(EXISTING_VIDEO)

with zipfile.ZipFile(EXISTING_ARCHIVE) as arc:
    existing_detections = json.loads(arc.read("detections.json"))

cap = cv2.VideoCapture(str(EXISTING_VIDEO))
existing_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
cap.release()

print(f"Loaded {len(existing_frames)} frames @ {existing_fps:.1f} fps")
print(f"Detections on {sum(1 for v in existing_detections.values() if v)} frames")

In [ ]:
# Upscale the existing decompressed video with CogVideoX + ROI overlay
cog_upscaler = CogVideoUpscaler(cfg.get("cogvideo_upscaling", {}))

upscaled = cog_upscaler.upscale_sequence(
    existing_frames,
    detections=existing_detections,
)

out_path = out_dir / "bird1_upscaled_cogvideo.mp4"
assemble_video(iter(upscaled), out_path, fps=existing_fps)
print(f"Saved → {out_path}")

show_frames(upscaled, n=5, figsize=(18, 4))

## 8 — CLI Equivalent

All of the above can also be run with a single shell command:

```bash
# Full pipeline
python run_pipeline.py \
  --input data/bird1.mp4 \
  --config configs/gpu/compression.yaml \
  --stages compress decompress restore upscale-cogvideo

# Just upscale an already-decompressed video, using saved ROI bboxes
python run_pipeline.py \
  --input      outputs/compression/bird1_decompressed.mp4 \
  --detections outputs/compression/bird1.zip \
  --config     configs/gpu/compression.yaml \
  --stages     upscale-cogvideo

# Bicubic only (fast, no GPU model needed)
python run_pipeline.py \
  --input  outputs/compression/bird1_decompressed.mp4 \
  --config configs/gpu/compression.yaml \
  --stages upscale-bicubic

# Generate Wan2.1 bookends (hallucinated full video from first/last frame)
python generate_wan_bookends.py \
  --input data/bird1.mp4 \
  --config configs/gpu/compression.yaml
```

## 9 — PSNR / SSIM Comparison

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity  as ssim

def compute_metrics(ref_frames, cmp_frames, n_sample=20):
    n = min(len(ref_frames), len(cmp_frames))
    idxs = np.linspace(0, n - 1, min(n_sample, n), dtype=int)
    psnrs, ssims = [], []
    for i in idxs:
        r = cv2.resize(ref_frames[i], (cmp_frames[i].shape[1], cmp_frames[i].shape[0]))
        r_rgb = cv2.cvtColor(r, cv2.COLOR_BGR2RGB)
        c_rgb = cv2.cvtColor(cmp_frames[i], cv2.COLOR_BGR2RGB)
        psnrs.append(psnr(r_rgb, c_rgb, data_range=255))
        ssims.append(ssim(r_rgb, c_rgb, channel_axis=2, data_range=255))
    return np.mean(psnrs), np.mean(ssims)

stages = {
    "Decompressed": decomp_frames,
    "Restored":     restored_frames,
    "Bicubic":      bicubic_frames,
    "CogVideoX":    cogvideo_frames,
    # "Wan2.1":     wan_frames,
}

print(f"{'Stage':<16} {'PSNR (dB)':>10} {'SSIM':>8}")
print("-" * 36)
for name, frames in stages.items():
    p, s = compute_metrics(raw_frames, frames)
    print(f"{name:<16} {p:>10.2f} {s:>8.4f}")

In [ ]:
# Bar chart
metrics = {name: compute_metrics(raw_frames, frames) for name, frames in stages.items()}
names   = list(metrics.keys())
psnr_v  = [metrics[n][0] for n in names]
ssim_v  = [metrics[n][1] for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(names, psnr_v, color="steelblue")
ax1.set_title("PSNR (dB) — higher is better")
ax1.set_ylabel("dB")

ax2.bar(names, ssim_v, color="tomato")
ax2.set_title("SSIM — higher is better")
ax2.set_ylabel("SSIM")

for ax in (ax1, ax2):
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()